# Session 7: Sequence Models

In [ ]:
import os
import math
import random
import time
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

Device: cpu


# Sample Dataset

In [ ]:
text = '''
To be, or not to be, that is the question:
Whether 'tis nobler in the mind to suffer
The slings and arrows of outrageous fortune,
Or to take arms against a sea of troubles,
And by opposing end them. To die—to sleep,
No more; and by a sleep to say we end
The heart-ache and the thousand natural shocks
That flesh is heir to: 'tis a consummation
Devoutly to be wish'd. To die, to sleep;'''

text = text.replace('\n', ' \n ')
chars = sorted(list(set(text)))
vocab = {ch:i for i,ch in enumerate(chars)}
ivocab = {i:ch for ch,i in vocab.items()}


# Dataset Class

In [ ]:
class CharDataset(Dataset):
    def __init__(self, text_indices, seq_len=30):
        self.data = torch.tensor(text_indices, dtype=torch.long)
        self.seq_len = seq_len
    def __len__(self):
        return len(self.data) - self.seq_len
    def __getitem__(self, idx):
        x = self.data[idx:idx+self.seq_len]
        y = self.data[idx+1:idx+self.seq_len+1]
        return x, y


encoded = [vocab[ch] for ch in text]
dataset = CharDataset(encoded)
loader = DataLoader(dataset, batch_size=64, shuffle=True, drop_last=True)


# Build the Model

In [ ]:
class CharRNN(nn.Module):
    def __init__(self, vocab_size, embed_size=128, hidden_size=256, num_layers=1, rnn_type='lstm'):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.rnn_type = rnn_type.lower()
        if self.rnn_type == 'rnn':
            self.rnn = nn.RNN(embed_size, hidden_size, num_layers=num_layers, batch_first=True)
        elif self.rnn_type == 'gru':
            self.rnn = nn.GRU(embed_size, hidden_size, num_layers=num_layers, batch_first=True)
        else:
            self.rnn = nn.LSTM(embed_size, hidden_size, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)
    def forward(self, x, hidden):
        x = self.embed(x)
        out, hidden = self.rnn(x, hidden)
        logits = self.fc(out)
        return logits, hidden

vocab_size = len(vocab)
model = CharRNN(vocab_size, embed_size=128, hidden_size=256, num_layers=2, rnn_type='lstm').to(device)
print(model)


CharRNN(
  (embed): Embedding(38, 128)
  (rnn): LSTM(128, 256, num_layers=2, batch_first=True)
  (fc): Linear(in_features=256, out_features=38, bias=True)
)


# Train

In [ ]:
def detach_hidden(hidden):
    if hidden is None:
        return None
    if isinstance(hidden, tuple):
        return (hidden[0].detach(), hidden[1].detach())
    else:
        return hidden.detach()

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

def train_epoch(model, loader, optimizer, clip=1.0):
    model.train()
    total_loss = 0.0
    hidden = None
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad()
        logits, hidden = model(xb, hidden)
        hidden = detach_hidden(hidden)
        loss = criterion(logits.view(-1, logits.size(-1)), yb.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

start = time.time()
loss = train_epoch(model, loader, optimizer)
print('Loss:', loss)

Loss: 3.0501882076263427


# Full Training

In [ ]:
def generate(model, start_string='T', length=200, temperature=1.0):
    model.eval()
    chars = [ch for ch in start_string]
    h = None
    x = torch.tensor([[vocab[chars[0]]]], dtype=torch.long).to(device)
    with torch.no_grad():
        for i in range(length):
            logits, h = model(x, h)
            logits = logits[:, -1, :] / max(1e-8, temperature)
            probs = torch.softmax(logits, dim=-1)
            idx = torch.multinomial(probs, num_samples=1).item()
            ch = ivocab[idx]
            chars.append(ch)
            x = torch.tensor([[idx]], dtype=torch.long).to(device)
    return ''.join(chars)

epochs = 10
for epoch in range(1, epochs+1):
    t0 = time.time()
    loss = train_epoch(model, loader, optimizer)
    t1 = time.time()
    sample = generate(model, start_string='To', length=200, temperature=0.8)
    print(sample)
    print("-----------------------------------")

Tohe question: 
 Whether 'tis nobler in the mind to suffer 
 To be, or not to be, that is the question: 
 Whether 'tis nobler in the mind to suffer 
 The slings and arrows of outrageous fortune, to be, 
-----------------------------------
To—the question: 
 Whether 'tis nobler in the mind to suffer 
 The slings and arrows of outrageous fortune, 
 Or to take artis nobler in the mind to suffer 
 The slings and arrows of outrageous fortune,
-----------------------------------
Toe outrageous fortune, 
 Or to take arms agearows of outrageous fore; and by a sleep to sa flesh is heir to: 'tis a consummation 
 Devoutly to be wish'd. To die, to sleep, 
 No more; and by a sleep to 
-----------------------------------
Toond by a sleep to say we end 
 The heart-ache and the thousand natural shocks 
 That flesh is heir to: 'tis a consummation 
 Devoutly to be wish'd. To die, to sleep to say we end 
 The heart-ache and 
-----------------------------------
Too to aumsagainst a sea of troubles, 
 And 

## Exercise

Build an LSTM language model on a real dataset. Deliverables:
- Cleaned dataset & tokenizer
- Training notebook with experiments (LR, grad clipping, optimizer choice)
- Generated samples
- Ablation study: compare RNN vs GRU vs LSTM, different hidden sizes, temperature
- Final report and short presentation

RuntimeError: duplicate registrations for aten.linspace.Tensor_Tensor

In [ ]:
# =========================
# Session 7 – Sequence Model (Simple & Clean)
# =========================

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import time

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:", device)

# =========================
# Simple Text Dataset (Custom)
# =========================

text = '''
He who seeks truth must first learn to doubt.
Man is something that must be overcome,
Not preserved in comfort and habit.
Out of chaos the strong spirit creates order,
And out of suffering meaning is born.
To follow the crowd is to sleep,
To walk alone is to awaken.
One must still have chaos within,
To give birth to a dancing star.
'''

text = text.lower()
text = text.replace('\n', ' \n ')

chars = sorted(list(set(text)))
vocab = {ch: i for i, ch in enumerate(chars)}
ivocab = {i: ch for ch, i in vocab.items()}
vocab_size = len(vocab)

encoded = [vocab[ch] for ch in text]

print("Vocab size:", vocab_size)

# =========================
# Dataset
# =========================

class CharDataset(Dataset):
    def __init__(self, data, seq_len=25):
        self.data = torch.tensor(data, dtype=torch.long)
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        x = self.data[idx:idx+self.seq_len]
        y = self.data[idx+1:idx+self.seq_len+1]
        return x, y

dataset = CharDataset(encoded, seq_len=25)
loader = DataLoader(dataset, batch_size=32, shuffle=True, drop_last=True)

# =========================
# Model (Same as Lecture)
# =========================

class CharRNN(nn.Module):
    def __init__(self, vocab_size, embed_size=64, hidden_size=128,
                 num_layers=1, rnn_type='lstm'):
        super().__init__()

        self.embed = nn.Embedding(vocab_size, embed_size)
        self.rnn_type = rnn_type.lower()

        if self.rnn_type == 'rnn':
            self.rnn = nn.RNN(embed_size, hidden_size,
                              num_layers=num_layers, batch_first=True)
        elif self.rnn_type == 'gru':
            self.rnn = nn.GRU(embed_size, hidden_size,
                              num_layers=num_layers, batch_first=True)
        else:
            self.rnn = nn.LSTM(embed_size, hidden_size,
                               num_layers=num_layers, batch_first=True)

        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden):
        x = self.embed(x)
        out, hidden = self.rnn(x, hidden)
        logits = self.fc(out)
        return logits, hidden

model = CharRNN(
    vocab_size=vocab_size,
    embed_size=64,
    hidden_size=128,
    num_layers=1,
    rnn_type='lstm'   # rnn | gru | lstm
).to(device)

print(model)

# =========================
# Training Setup
# =========================

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

def detach_hidden(h):
    if h is None:
        return None
    if isinstance(h, tuple):
        return (h[0].detach(), h[1].detach())
    return h.detach()

def train_epoch(model, loader):
    model.train()
    total_loss = 0
    hidden = None

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        logits, hidden = model(xb, hidden)
        hidden = detach_hidden(hidden)

        loss = criterion(
            logits.view(-1, vocab_size),
            yb.view(-1)
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

# =========================
# Text Generation
# =========================

def generate(model, start='d', length=150, temperature=0.8):
    model.eval()
    result = list(start)
    hidden = None

    x = torch.tensor([[vocab[start]]], device=device)

    with torch.no_grad():
        for _ in range(length):
            logits, hidden = model(x, hidden)
            logits = logits[:, -1, :] / temperature
            probs = torch.softmax(logits, dim=-1)
            idx = torch.multinomial(probs, 1).item()

            result.append(ivocab[idx])
            x = torch.tensor([[idx]], device=device)

    return ''.join(result)

# =========================
# Train
# =========================

epochs = 10
for epoch in range(1, epochs + 1):
    loss = train_epoch(model, loader)
    print(f"Epoch {epoch} | Loss: {loss:.4f}")
    print(generate(model, start='d', temperature=0.8))
    print("-" * 50)


In [8]:
print("Attempting to resolve PyTorch installation issues...")

# Uninstall current PyTorch, torchvision, and torchaudio
!pip uninstall torch torchvision torchaudio -y

# Install a known stable version of PyTorch compatible with Colab's typical CUDA setup.
# This specific command targets CUDA 12.1. If you encounter further issues,
# you might need to adjust the CUDA version in the URL.
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

print("PyTorch and torchvision reinstallation initiated. \n\n*************************************************************")
print("** IMPORTANT: Please RESTART THE RUNTIME NOW (Runtime -> Restart runtime). **")
print("** After restarting, run ALL cells in your notebook from the beginning. **")
print("*************************************************************")

Attempting to resolve PyTorch installation issues...
Found existing installation: torch 2.5.1+cu121
Uninstalling torch-2.5.1+cu121:
  Successfully uninstalled torch-2.5.1+cu121
Found existing installation: torchvision 0.20.1+cu121
Uninstalling torchvision-0.20.1+cu121:
  Successfully uninstalled torchvision-0.20.1+cu121
Found existing installation: torchaudio 2.5.1+cu121
Uninstalling torchaudio-2.5.1+cu121:
  Successfully uninstalled torchaudio-2.5.1+cu121
Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download.pytorch.org/whl/cu121/torch-2.5.1%2Bcu121-cp312-cp312-linux_x86_64.whl (780.4 MB)
  Using cached https://download.pytorch.org/whl/cu121/torchvision-0.20.1%2Bcu121-cp312-cp312-linux_x86_64.whl (7.3 MB)
  Using cached https://download.pytorch.org/whl/cu121/torchaudio-2.5.1%2Bcu121-cp312-cp312-linux_x86_64.whl (3.4 MB)


PyTorch and torchvision reinstallation initiated. 

*************************************************************
** IMPORTANT: Please RESTART THE RUNTIME NOW (Runtime -> Restart runtime). **
** After restarting, run ALL cells in your notebook from the beginning. **
*************************************************************


In [2]:
# =========================
# Session 7 – Sequence Model (Simple & Clean)
# =========================

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import time

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:", device)

# =========================
# Simple Text Dataset (Custom)
# =========================

text = '''
He who seeks truth must first learn to doubt.
Man is something that must be overcome,
Not preserved in comfort and habit.
Out of chaos the strong spirit creates order,
And out of suffering meaning is born.
To follow the crowd is to sleep,
To walk alone is to awaken.
One must still have chaos within,
To give birth to a dancing star.
'''

text = text.lower()
text = text.replace('\n', ' \n ')

chars = sorted(list(set(text)))
vocab = {ch: i for i, ch in enumerate(chars)}
ivocab = {i: ch for ch, i in vocab.items()}
vocab_size = len(vocab)

encoded = [vocab[ch] for ch in text]

print("Vocab size:", vocab_size)

# =========================
# Dataset
# =========================

class CharDataset(Dataset):
    def __init__(self, data, seq_len=25):
        self.data = torch.tensor(data, dtype=torch.long)
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        x = self.data[idx:idx+self.seq_len]
        y = self.data[idx+1:idx+self.seq_len+1]
        return x, y

dataset = CharDataset(encoded, seq_len=25)
loader = DataLoader(dataset, batch_size=32, shuffle=True, drop_last=True)

# =========================
# Model (Same as Lecture)
# =========================

class CharRNN(nn.Module):
    def __init__(self, vocab_size, embed_size=64, hidden_size=128,
                 num_layers=1, rnn_type='lstm'):
        super().__init__()

        self.embed = nn.Embedding(vocab_size, embed_size)
        self.rnn_type = rnn_type.lower()

        if self.rnn_type == 'rnn':
            self.rnn = nn.RNN(embed_size, hidden_size,
                              num_layers=num_layers, batch_first=True)
        elif self.rnn_type == 'gru':
            self.rnn = nn.GRU(embed_size, hidden_size,
                              num_layers=num_layers, batch_first=True)
        else:
            self.rnn = nn.LSTM(embed_size, hidden_size,
                               num_layers=num_layers, batch_first=True)

        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden):
        x = self.embed(x)
        out, hidden = self.rnn(x, hidden)
        logits = self.fc(out)
        return logits, hidden

model = CharRNN(
    vocab_size=vocab_size,
    embed_size=64,
    hidden_size=128,
    num_layers=1,
    rnn_type='lstm'   # rnn | gru | lstm
).to(device)

print(model)

# =========================
# Training Setup
# =========================

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

def detach_hidden(h):
    if h is None:
        return None
    if isinstance(h, tuple):
        return (h[0].detach(), h[1].detach())
    return h.detach()

def train_epoch(model, loader):
    model.train()
    total_loss = 0
    hidden = None

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        logits, hidden = model(xb, hidden)
        hidden = detach_hidden(hidden)

        loss = criterion(
            logits.view(-1, vocab_size),
            yb.view(-1)
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

# =========================
# Text Generation
# =========================

def generate(model, start='d', length=150, temperature=0.8):
    model.eval()
    result = list(start)
    hidden = None

    x = torch.tensor([[vocab[start]]], device=device)

    with torch.no_grad():
        for _ in range(length):
            logits, hidden = model(x, hidden)
            logits = logits[:, -1, :] / temperature
            probs = torch.softmax(logits, dim=-1)
            idx = torch.multinomial(probs, 1).item()

            result.append(ivocab[idx])
            x = torch.tensor([[idx]], device=device)

    return ''.join(result)

# =========================
# Train
# =========================

epochs = 40
for epoch in range(1, epochs + 1):
    loss = train_epoch(model, loader)
    print(f"Epoch {epoch} | Loss: {loss:.4f}")
    print(generate(model, start='d', temperature=0.8))
    print("-" * 50)


Device: cuda
Vocab size: 25
CharRNN(
  (embed): Embedding(25, 64)
  (rnn): LSTM(64, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=25, bias=True)
)
Epoch 1 | Loss: 3.1305
dneivh,wckdasii.tfvor.nlpav
uge.ksvdgt.b 
tnwhhlofm  ssit gadhvcvmof,rwls vofanvuhkov ,wbtv
.at ,m.sgipvegsvn .obn 

ti..mtkfu,gaud hongelkcgrc uvthsa.
--------------------------------------------------
Epoch 2 | Loss: 2.8109
d
 stve o   o   mr ue t t    t,u  e nvr  a s mu toer a.b
ont reeo owo inuto   tb   mosooi st ta   anen te   lbrfto witf e ce we . rttoa  toenw tso i   
--------------------------------------------------
Epoch 3 | Loss: 2.5932
d  ir onm vdwlnph sn  b
 oseda ins h tf a bsoevs aret  
uowrtb ow ,t.  ean o l mte o nlo e  ot t o an odean an s.mvneref ogsrst ohng ioa oupt e poron. 
--------------------------------------------------
Epoch 4 | Loss: 2.3890
daite a h oo dr os,temmi mtt oe taun  ohri
nh toe mo oan t otw oon te . io sot ofiste cto w ot mout nl lue oao set  
 esorwe fio
 